# Tamil BPE Tokenizer - Using Real HuggingFace Dataset

**Language**: Tamil (தமிழ்)
**Requirements**: 5000+ vocab, 3.0+ compression

In [1]:
import os
import json
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers, normalizers
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

print("✅ All libraries imported!")

✅ All libraries imported!


In [2]:
# Download REAL Tamil dataset from HuggingFace
print("Downloading REAL Tamil dataset from HuggingFace...\n")

dataset = None

# Try AI4Bharat IndicCorp v2 - Large Tamil corpus
try:
    print("[1/3] Trying AI4Bharat IndicCorp Tamil (BEST OPTION)...")
    print("      This is a large, real Tamil text corpus\n")
    
    dataset = load_dataset(
        "ai4bharat/IndicParaphrase",
        "ta",
        split="train",
        trust_remote_code=True
    )
    
    # Take first 100K for training (enough for 5K+ vocab)
    if len(dataset) > 100000:
        dataset = dataset.select(range(100000))
    
    print(f"✅ SUCCESS! Loaded {len(dataset):,} real Tamil texts")
    print(f"   Source: AI4Bharat IndicParaphrase\n")
    
except Exception as e:
    print(f"   Failed: {e}\n")
    dataset = None

# Try Tamil subset of Multilingual dataset
if dataset is None:
    try:
        print("[2/3] Trying Multilingual Tamil dataset...")
        
        dataset = load_dataset(
            "facebook/flores",
            "tam_Taml",
            split="devtest"
        )
        
        # This is smaller, so we'll repeat it to get enough data
        print(f"   Loaded {len(dataset)} samples, replicating for training...")
        
        # Replicate dataset 1000x to get enough training data
        from datasets import concatenate_datasets
        datasets_list = [dataset] * 1000
        dataset = concatenate_datasets(datasets_list)
        
        print(f"✅ SUCCESS! Created {len(dataset):,} Tamil texts\n")
        
    except Exception as e:
        print(f"   Failed: {e}\n")
        dataset = None

# Try Wikipedia Tamil (most reliable)
if dataset is None:
    try:
        print("[3/3] Trying Tamil Wikipedia (MOST RELIABLE)...")
        
        # Use streaming to download progressively
        dataset_stream = load_dataset(
            "wikimedia/wikipedia",
            "20231101.ta",
            split="train",
            streaming=True
        )
        
        print("   Downloading Tamil Wikipedia articles...")
        
        # Take first 50K articles
        articles = []
        for i, article in enumerate(dataset_stream):
            if i >= 50000:
                break
            if article['text'].strip():
                articles.append(article)
            if (i+1) % 10000 == 0:
                print(f"   Downloaded {i+1:,} articles...")
        
        from datasets import Dataset
        dataset = Dataset.from_list(articles)
        
        print(f"\n✅ SUCCESS! Loaded {len(dataset):,} real Tamil Wikipedia articles\n")
        
    except Exception as e:
        print(f"   Failed: {e}\n")
        dataset = None

if dataset is None:
    raise Exception(
        "\n❌ Could not load any Tamil dataset!\n"
        "Please check internet connection or try:\n"
        "  huggingface-cli login"
    )

# Show sample
print("="*70)
print("REAL TAMIL DATASET LOADED")
print("="*70)
sample = dataset[0]['text'] if 'text' in dataset[0] else str(dataset[0])
print(f"\nSample Tamil text ({len(sample)} chars):")
print(sample[:300])
if len(sample) > 300:
    print("...\n")

print(f"Total documents: {len(dataset):,}")
print(f"This is REAL Tamil language data from HuggingFace! ✅")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ai4bharat/IndicParaphrase' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.



[1/3] Trying AI4Bharat IndicCorp Tamil (BEST OPTION)...
      This is a large, real Tamil text corpus

   Failed: Dataset scripts are no longer supported, but found IndicParaphrase.py

[2/3] Trying Multilingual Tamil dataset...
   Failed: Dataset scripts are no longer supported, but found flores.py

[3/3] Trying Tamil Wikipedia (MOST RELIABLE)...
   Downloaded 10,000 articles...
   Downloaded 20,000 articles...
   Downloaded 30,000 articles...
   Downloaded 40,000 articles...
   Downloaded 50,000 articles...

✅ SUCCESS! Loaded 50,000 real Tamil Wikipedia articles

REAL TAMIL DATASET LOADED

Sample Tamil text (22 chars):
விக்கிப்பீடியா மொழிகள்
Total documents: 50,000
This is REAL Tamil language data from HuggingFace! ✅


In [3]:
# Prepare evaluation texts
eval_texts = []
for i in range(min(100, len(dataset))):
    text = dataset[i]['text'] if 'text' in dataset[i] else str(dataset[i])
    if text.strip():
        eval_texts.append(text)

print(f"✅ Prepared {len(eval_texts)} texts for evaluation")
print(f"📊 Total dataset: {len(dataset):,} documents")

✅ Prepared 100 texts for evaluation
📊 Total dataset: 50,000 documents


In [4]:
# Initialize BPE tokenizer
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))

tokenizer.normalizer = normalizers.Sequence([
    normalizers.NFD(),
    normalizers.StripAccents()
])

tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

print("✅ Tokenizer initialized")

✅ Tokenizer initialized


In [5]:
# Configure trainer with 8000 vocab (exceeds 5000 requirement)
vocab_size = 8000

trainer = trainers.BpeTrainer(
    vocab_size=vocab_size,
    special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"],
    min_frequency=2,
    show_progress=True
)

print(f"✅ Trainer configured")
print(f"   Target vocabulary: {vocab_size:,} tokens")
print(f"   Requirement: 5,000+ tokens ✅")

✅ Trainer configured
   Target vocabulary: 8,000 tokens
   Requirement: 5,000+ tokens ✅


In [6]:
# Train tokenizer
print("🚀 Training BPE tokenizer...")
print("   This will take 10-20 minutes\n")

def get_training_corpus(batch_size=1000):
    """Generator that yields batches of text."""
    for i in range(0, len(dataset), batch_size):
        # Get batch - HF datasets return dict of lists
        end_idx = min(i + batch_size, len(dataset))
        
        texts = []
        for j in range(i, end_idx):
            # Access individual items
            item = dataset[j]
            text = item['text'] if isinstance(item, dict) and 'text' in item else str(item)
            if text and text.strip():
                texts.append(text)
        
        if texts:
            yield texts

tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)

print("\n✅ Training completed!")

🚀 Training BPE tokenizer...
   This will take 10-20 minutes





✅ Training completed!


In [11]:
# Check vocabulary
vocab = tokenizer.get_vocab()
actual_vocab_size = len(vocab)

print("="*70)
print("VOCABULARY CHECK")
print("="*70)
print(f"Target vocab: {vocab_size:,}")
print(f"Actual vocab: {actual_vocab_size:,}")
print(f"Requirement (5000+): {'✅ PASSED' if actual_vocab_size >= 5000 else '❌ FAILED'}")
print("="*70)

VOCABULARY CHECK
Target vocab: 8,000
Actual vocab: 8,000
Requirement (5000+): ✅ PASSED


In [12]:
# Test tokenization
test_texts = [
    "வணக்கம், இது தமிழ் மொழி பயிற்சி உதாரணம்",
    "தமிழ் இந்தியாவின் பழமையான மொழிகளில் ஒன்று",
    "செயற்கை நுண்ணறிவு என்பது மிகவும் சுவாரஸ்யமான துறை"
]

print("🧪 Testing tokenizer:\n")
for i, text in enumerate(test_texts, 1):
    encoding = tokenizer.encode(text)
    print(f"Example {i}:")
    print(f"  Text: {text}")
    print(f"  Tokens: {encoding.tokens[:10]}...")
    print(f"  Count: {len(encoding.tokens)} tokens\n")

🧪 Testing tokenizer:

Example 1:
  Text: வணக்கம், இது தமிழ் மொழி பயிற்சி உதாரணம்
  Tokens: ['à®µà®£', 'à®ķà®ķà®®', ',', 'Ġà®ĩà®¤', 'Ġà®¤à®®à®´', 'Ġà®®à®´', 'Ġà®ªà®¯à®±à®ļ', 'Ġà®īà®¤à®°à®£à®®']...
  Count: 8 tokens

Example 2:
  Text: தமிழ் இந்தியாவின் பழமையான மொழிகளில் ஒன்று
  Tokens: ['à®¤à®®à®´', 'Ġà®ĩà®¨à®¤à®¯à®µà®©', 'Ġà®ªà®´à®®à®¯à®©', 'Ġà®®à®´à®ķà®³à®²', 'Ġà®Ĵà®©à®±']...
  Count: 5 tokens

Example 3:
  Text: செயற்கை நுண்ணறிவு என்பது மிகவும் சுவாரஸ்யமான துறை
  Tokens: ['à®ļà®¯', 'à®±à®ķ', 'Ġà®¨à®£à®£', 'à®±à®µ', 'Ġà®İà®©à®ªà®¤', 'Ġà®®à®ķà®µà®®', 'Ġà®ļà®µà®°', 'à®¸', 'à®¯à®®à®©', 'Ġà®¤à®±']...
  Count: 10 tokens



In [9]:
# Calculate compression ratio
print("📈 Calculating compression ratio...\n")

total_chars = 0
total_tokens = 0

for text in tqdm(eval_texts, desc="Evaluating"):
    total_chars += len(text)
    encoding = tokenizer.encode(text)
    total_tokens += len(encoding.tokens)

compression_ratio = total_chars / total_tokens if total_tokens > 0 else 0

print("\n" + "="*70)
print("COMPRESSION CHECK")
print("="*70)
print(f"Total characters: {total_chars:,}")
print(f"Total tokens: {total_tokens:,}")
print(f"Compression ratio: {compression_ratio:.4f}")
print(f"Requirement (≥ 3.0): {'✅ PASSED' if compression_ratio >= 3.0 else '❌ FAILED'}")
print("="*70)

📈 Calculating compression ratio...



Evaluating: 100%|██████████| 100/100 [00:00<00:00, 388.94it/s]


COMPRESSION CHECK
Total characters: 582,608
Total tokens: 124,834
Compression ratio: 4.6671
Requirement (≥ 3.0): ✅ PASSED


In [10]:
# Save tokenizer
tokenizer.save("tamil_bpe_tokenizer.json")

summary = {
    "language": "Tamil",
    "algorithm": "BPE",
    "vocabulary_size": actual_vocab_size,
    "compression_ratio": round(compression_ratio, 4),
    "meets_vocab_requirement": actual_vocab_size >= 5000,
    "meets_compression_requirement": compression_ratio >= 3.0,
    "dataset_size": len(dataset),
    "dataset_source": "HuggingFace (Real Tamil Data)"
}

with open('tokenizer_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)
for key, value in summary.items():
    print(f"{key.replace('_', ' ').title()}: {value}")
print("="*70)

if summary['meets_vocab_requirement'] and summary['meets_compression_requirement']:
    print("\n✅ ALL REQUIREMENTS MET! 🎉")
    print("\nFiles created:")
    print("  - tamil_bpe_tokenizer.json")
    print("  - tokenizer_summary.json")
else:
    print("\n⚠️ Some requirements not met")


FINAL SUMMARY
Language: Tamil
Algorithm: BPE
Vocabulary Size: 8000
Compression Ratio: 4.6671
Meets Vocab Requirement: True
Meets Compression Requirement: True
Dataset Size: 50000
Dataset Source: HuggingFace (Real Tamil Data)

✅ ALL REQUIREMENTS MET! 🎉

Files created:
  - tamil_bpe_tokenizer.json
  - tokenizer_summary.json
